# 00 — Environment Setup & Project Scaffolding
### Multimodal Deepfake Detection — Image + Video + Audio (Feature Fusion Project)

This notebook sets up everything the other three notebooks depend on:

- Installs required libraries
- Creates the folder structure for data, checkpoints, and extracted features
- Verifies GPU availability
- Defines the **dataset folder convention** every other notebook expects

**Run this notebook first, once, before `01_image_efficientnet_b4.ipynb`, `02_video_cnn_lstm.ipynb`, or `03_audio_wavlm.ipynb`.**

> Reference: this matches Phase 1–3 of the project's implementation plan (individual modality
> branches trained/fine-tuned independently before feature fusion, which is Phase 4–5, done later).

## 1. Install dependencies
Run once per environment (Colab / Kaggle / local venv).

In [ ]:
# If running on Colab / a fresh environment, uncomment and run:
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install -q timm transformers librosa soundfile opencv-python-headless scikit-learn matplotlib tqdm pandas

import importlib
required = ["torch", "torchvision", "timm", "transformers", "cv2", "librosa", "soundfile", "sklearn", "matplotlib", "tqdm", "pandas"]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print("Missing packages (install these before continuing):", missing)
else:
    print("All required packages are importable. You're good to go.")

## 2. Check hardware

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected — the notebooks will still run (synthetic-data smoke tests are fast on CPU),")
    print("but real fine-tuning on full datasets will be slow. Use Colab/Kaggle GPU runtime if needed.")

## 3. Project folder structure

Every notebook expects this layout, relative to the project root (adjust `PROJECT_ROOT` below if needed).
Create it now so paths resolve correctly in every notebook:

```
project_root/
├── data/
│   ├── image/
│   │   ├── real/        <- real face images (.jpg/.png)
│   │   └── fake/        <- deepfake/manipulated images
│   ├── video/
│   │   ├── real/        <- real videos (.mp4)
│   │   └── fake/        <- deepfake videos
│   └── audio/
│       ├── real/        <- genuine speech (.wav)
│       └── fake/         <- synthetic/cloned speech
├── checkpoints/          <- fine-tuned model weights get saved here
├── features/             <- extracted F_image / F_video / F_audio tensors (for the fusion stage, later)
└── notebooks/            <- this notebook and the other 3 live here
```

You don't need real data to execute the notebooks today — each one falls back to a small
synthetic dataset so you can validate the full pipeline end-to-end first, then drop in a
real dataset (e.g. FaceForensics++, DFDC, Celeb-DF for video/image; ASVspoof / WaveFake for audio)
later without changing any code.

In [ ]:
import os

PROJECT_ROOT = "."  # adjust if your notebooks/ folder is nested differently
DIRS = [
    "data/image/real", "data/image/fake",
    "data/video/real", "data/video/fake",
    "data/audio/real", "data/audio/fake",
    "checkpoints",
    "models",
    "features",
]

for d in DIRS:
    path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(path, exist_ok=True)

print("Project scaffolding ready under:", os.path.abspath(PROJECT_ROOT))
for d in DIRS:
    print(" -", os.path.join(PROJECT_ROOT, d))

## 4. Shared config

A single place for constants every notebook re-uses (image size, sample rate, number of frames, etc.),
so the three branches stay consistent with each other — this matters later because the fusion network
needs feature vectors that were produced under matching preprocessing assumptions.

In [ ]:
import json

SHARED_CONFIG = {
    "project_root": PROJECT_ROOT,
    "seed": 42,
    "image": {"img_size": 380, "batch_size": 16, "backbone": "tf_efficientnet_b4_ns"},
    "video": {"num_frames": 16, "frame_size": 224, "batch_size": 4, "cnn_feature_dim": 512, "lstm_hidden": 256},
    "audio": {"sample_rate": 16000, "max_audio_seconds": 4, "batch_size": 8, "wavlm_checkpoint": "microsoft/wavlm-base-plus"},
    "fusion": {"projection_dim": 256}  # used later, in the feature-fusion notebook
}

with open(os.path.join(PROJECT_ROOT, "shared_config.json"), "w") as f:
    json.dump(SHARED_CONFIG, f, indent=2)

print(json.dumps(SHARED_CONFIG, indent=2))

## Next steps

1. Run `01_image_efficientnet_b4.ipynb` — image branch (EfficientNet-B4)
2. Run `02_video_cnn_lstm.ipynb` — video branch (CNN + LSTM)
3. Run `03_audio_wavlm.ipynb` — audio branch (WavLM)
4. Once all three branches run cleanly (even on synthetic data), drop your real dataset into
   `data/image`, `data/video`, `data/audio` following the folder convention above, and re-run.
5. Feature fusion (`04_feature_extraction.ipynb` + `05_feature_fusion_training.ipynb`) comes after
   — that's a separate, later stage as you asked.